Setups and imports

In [3]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split


sys.path.append(os.path.abspath('scripts'))

from data_loader import SODDataset
from sod_model import ImprovedSOD, BaselineSOD
from train import train_model
from evaluate import iou, f1, precision, recall

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Duke perdorur paisjen: {device}")

ModuleNotFoundError: No module named 'data_loader'

Dataset Download

In [ ]:
!wget http://saliencydetection.net/duts/download/DUTS-TR.zip
!wget http://saliencydetection.net/duts/download/DUTS-TE.zip
!unzip -q DUTS-TR.zip
!unzip -q DUTS-TE.zip

img_dir = "DUTS-TR/DUTS-TR-Image"
mask_dir = "DUTS-TR/DUTS-TR-Mask"

if os.path.exists(img_dir) and os.path.exists(mask_dir):
    print("Imazhe te gjetura:", len(os.listdir(img_dir)))
    print("Maska te gjetura:", len(os.listdir(mask_dir)))
else:
    print("GABIM: Folderat nuk u gjeten!")

Train,validation and split data

In [2]:
all_imgs = sorted(os.listdir(img_dir))

train_list, temp_list = train_test_split(
    all_imgs,
    test_size=0.3,
    random_state=42
)

val_list, test_list = train_test_split(
    temp_list,
    test_size=0.5,
    random_state=42
)

print(f"Training images: {len(train_list)}")
print(f"Validation images: {len(val_list)}")
print(f"Test images: {len(test_list)}")

NameError: name 'img_dir' is not defined

Create Dataset Objects and Dataloaders


In [ ]:
train_dataset = SODDataset(img_dir, mask_dir, train_list)
val_dataset = SODDataset(img_dir, mask_dir, val_list)
test_dataset = SODDataset(img_dir, mask_dir, test_list)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

Initialize and Train Improved Model

In [ ]:
model = ImprovedSOD().to(device)
epochs = 20


model, train_losses, val_losses = train_model(model, train_loader, val_loader, device, epochs=epochs)

torch.save(model.state_dict(), "sod_model.pth")
print("Model saved successfully!")

Plot training and validation loss

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

Evaluate Model on Test Set

In [ ]:
model.eval()
total_iou, total_precision, total_recall, total_f1 = 0, 0, 0, 0

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        outputs = torch.sigmoid(outputs)

        total_iou += iou(outputs, masks).item()
        total_precision += precision(outputs, masks).item()
        total_recall += recall(outputs, masks).item()
        total_f1 += f1(outputs, masks).item()

print("Test Results")
print("------------------")
print(f"IoU: {total_iou/len(test_loader):.4f}")
print(f"Precision: {total_precision/len(test_loader):.4f}")
print(f"Recall: {total_recall/len(test_loader):.4f}")
print(f"F1 Score: {total_f1/len(test_loader):.4f}")

Final Prediction and Overlay Visualization

In [ ]:
model.eval()

image, mask = test_dataset[5]

with torch.no_grad():
    prediction = model(image.unsqueeze(0).to(device))
    prediction = torch.sigmoid(prediction)

prediction_np = prediction.squeeze().cpu().numpy()
binary_pred = (prediction_np > 0.4).astype(np.float32)

image_np = image.permute(1,2,0).numpy()
mask_np = mask.squeeze().numpy()


overlay = image_np.copy()
overlay[:,:,0] = overlay[:,:,0] + binary_pred * 0.5
overlay = np.clip(overlay, 0, 1)

plt.figure(figsize=(15,5))
plt.subplot(1,4,1); plt.imshow(image_np); plt.title("Input Image")
plt.subplot(1,4,2); plt.imshow(mask_np, cmap='gray'); plt.title("Ground Truth")
plt.subplot(1,4,3); plt.imshow(binary_pred, cmap='gray'); plt.title("Predicted Mask")
plt.subplot(1,4,4); plt.imshow(overlay); plt.title("Overlay")
plt.show()

Demo from other image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch


try:
    from google.colab import files
    uploaded = files.upload()
    test_img_path = list(uploaded.keys())[0]
    print(f"U ngarkua imazhi: {test_img_path}")
except:

    test_img_path = "foto_test.jpg"
    print(f"Duke perdorur path-in lokal: {test_img_path}")


image_raw = cv2.imread(test_img_path)
if image_raw is not None:
    image_rgb = cv2.cvtColor(image_raw, cv2.COLOR_BGR2RGB)
    image_res = cv2.resize(image_rgb, (128, 128)) / 255.0

    img_tensor = torch.tensor(image_res, dtype=torch.float32).permute(2,0,1).unsqueeze(0).to(device)


    model.eval()
    with torch.no_grad():
        pred = torch.sigmoid(model(img_tensor))
        pred = pred.squeeze().cpu().numpy()
        binary_m = (pred > 0.4).astype(np.float32)


    plt.figure(figsize=(6,6))
    plt.imshow(image_res)
    plt.imshow(binary_m, cmap='jet', alpha=0.5)
    plt.title("User Image Demo (Prediction Overlay)")
    plt.axis('off')
    plt.show()
else:
    print("GABIM: Imazhi nuk u gjet. Kontrollo path-in!")

Model Comparison Summary

In [ ]:
print("FINAL PROJECT RESULTS")
print("---------------------")
print("Improved Model:")
print(f"IoU: {total_iou/len(test_loader):.4f}")
print(f"F1: {total_f1/len(test_loader):.4f}")
print("---------------------")
print("Conclusion: Improved CNN significantly outperforms baseline model.")